# Chapter 4 — The Debugging Stack

**Book alignment:** Debugging AI From First Principles, Chapter 4

**Question this notebook isolates:** One `KeyError: 'refund_id'`, three layers that could
own it (Code assumption / Data regression / Environment drift). Does a layer-swap probe —
hold everything constant, swap one layer — eliminate layers cheapest-first and convict
exactly one?

In [ ]:
# clean -> build_row, with one defect flag per layer. Everything else is held constant.
def clean(orders, *, code_broke_propagation=False):
    out = []
    for o in orders:
        row = {"order_id": o["order_id"], "subtotal": o["subtotal"]}
        # spec: propagate refund_id to every row (incl. split-shipment children)
        if "refund_id" in o and not (code_broke_propagation and o.get("is_child")):
            row["refund_id"] = o["refund_id"]
        out.append(row)
    return out

def build_row(row):
    return {**row, "refund_id": row["refund_id"]}      # raises KeyError if the key is absent

def run(orders, *, code_broke_propagation=False):
    return [build_row(r) for r in clean(orders, code_broke_propagation=code_broke_propagation)]

GOOD_SNAPSHOT = [
    {"order_id": 1, "subtotal": 100.0, "refund_id": "RB-1"},
    {"order_id": 2, "subtotal": 200.0, "refund_id": "RB-2", "is_child": True},
]
BAD_SNAPSHOT = [{k: v for k, v in o.items() if k != "refund_id"} for o in GOOD_SNAPSHOT]  # export lost the column

def load(snapshot, *, env_dtype_bug=False):
    rows = [dict(o) for o in snapshot]
    if env_dtype_bug:                                  # 2.x CSV reader: whole column parses as NaN
        for r in rows:
            r.pop("refund_id", None)
    return rows

## 1. Three engineers, three fixes, one traceback

Each defect below raises the *identical* `KeyError: 'refund_id'`.

In [ ]:
def crashes(fn):
    try:
        fn(); return None
    except KeyError as e:
        return f"KeyError: {e}"

incidents = {
    "H1 code": dict(data=GOOD_SNAPSHOT, code_broke=True,  env_bug=False),   # V dropped child propagation
    "H2 data": dict(data=BAD_SNAPSHOT,  code_broke=False, env_bug=False),   # export lost the column
    "H3 env":  dict(data=GOOD_SNAPSHOT, code_broke=False, env_bug=True),    # same file, 2.x parses it differently
}
def replay(inc):
    return run(load(inc["data"], env_dtype_bug=inc["env_bug"]), code_broke_propagation=inc["code_broke"])

for name, inc in incidents.items():
    print(f"{name:8} -> {crashes(lambda: replay(inc))}")
assert len({crashes(lambda: replay(inc)) for inc in incidents.values()}) == 1
print("\nsame symptom, three layers, three owners - opinions until you have a layer order")

## 2. The layer-swap probe: one swapped layer per experiment, cheapest first

- **Code** probe: run this incident's data on the *previous* code version.
- **Data** probe: run the current code on the known-good snapshot.
- **Environment** probe: run with the parsing drift disabled.

In [ ]:
def diagnose(inc):
    # Code probe: V-1 (no code defect) on this incident's exact data + env
    if crashes(lambda: run(load(inc["data"], env_dtype_bug=inc["env_bug"]), code_broke_propagation=False)) is None:
        return "Code"
    # Data probe: current code on the KNOWN-GOOD snapshot (env held at incident's value)
    if crashes(lambda: run(load(GOOD_SNAPSHOT, env_dtype_bug=inc["env_bug"]),
                           code_broke_propagation=inc["code_broke"])) is None:
        return "Data"
    # Environment probe: incident's data + code, parsing drift disabled
    if crashes(lambda: run(load(inc["data"], env_dtype_bug=False),
                           code_broke_propagation=inc["code_broke"])) is None:
        return "Environment"
    return "Intent - UNKNOWN"

for name, inc in incidents.items():
    print(f"{name:8} -> {diagnose(inc)}")
assert [diagnose(incidents[k]) for k in ("H1 code", "H2 data", "H3 env")] == ["Code", "Data", "Environment"]
print("\none swapped layer per probe; the first swap that flips the outcome owns the fix")

## 3. The convicted layer routes the fix — and a `.get()` silence would corrupt, not fix

In [ ]:
# the code expert's instinct: guard build_row with .get(). It stops the crash ...
def build_row_silenced(row):
    return {**row, "refund_id": row.get("refund_id", "")}

silenced = [build_row_silenced(r) for r in clean(load(GOOD_SNAPSHOT), code_broke_propagation=True)]
child = next(r for r in silenced if r["order_id"] == 2)
print("silenced child row:", child)
assert child["refund_id"] == ""                         # ... by writing a blank refund into the report
print("loud crash -> silent corruption. assert at the handoff instead; convict the layer first.")

## What we earned

Three engineers proposed three fixes at three layers for one `KeyError`. The layer-swap
probe — hold everything constant, swap exactly one layer, cheapest first (Code → Data →
Environment → Intent) — convicts exactly one and suspends the rest. A `.get()` guard added
before the layer is known converts the loud crash into 12,000 silently wrong rows a night.

**Notebook 05 / Chapter 5** zooms into the Code layer: reading a Python traceback in two
passes to separate the frame that *raised* from the frame that *diverged first*.